# Skin Lesion Classification using Deep Residual Networks
**Student Name:** Osama Tarek Ghanem Eltokhy Rezk

**Matric No.:** 12453050

**Project:** HAM10000 Skin Cancer Detection


## 1. Environment Setup & Reproducibility
Importing necessary libraries, configuring hyperparameters, and setting the random seed to ensure results are identical across different machines (PC/Laptop).

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix
import wandb

KeyboardInterrupt: 

## 2. Configuration & Reproducibility
Setting hyperparameters and fixing the random seed to ensure that results are identical (reproducible) across your PC and Laptop.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device being used: {DEVICE}")

NUM_EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 1e-5
SEED = 42

CSV_PATH = 'HAM10000_metadata.csv'  
IMG_DIR = 'images' 

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# NEW: Initialize Weights & Biases
wandb.login() 
wandb.init(
    project="skin-lesion-classification", 
    name="resnet50-finetuning",
    config={
        "learning_rate": LEARNING_RATE,
        "architecture": "ResNet50",
        "dataset": "HAM10000",
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "seed": SEED
    }
)

Device being used: cpu


## 3. Data Loading
Defining the dataset class to handle image loading and label mapping.

In [ ]:
class HAM10000Dataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.metadata = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.label_map = {'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3, 'akiec': 4, 'vasc': 5, 'df': 6}

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.metadata.iloc[idx, 1] + '.jpg')
        image = Image.open(img_name).convert('RGB')
        label = self.label_map[self.metadata.iloc[idx, 2]]
        
        if self.transform:
            image = self.transform(image)
        return image, label

## 4. Transformations & Splitting
Applying **Data Augmentation** (Flip, Rotate, Color Jitter) while keeping validation data clean.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = HAM10000Dataset(CSV_PATH, IMG_DIR) 
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(full_dataset, [train_size, val_size], generator=generator)

train_subset.dataset.transform = train_transforms
val_subset.dataset.transform = val_transforms 

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Ready: {len(train_subset)} Train images | {len(val_subset)} Validation images")

Ready: 8012 Train images | 2003 Validation images


In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(2048, 7) 
model = model.to(DEVICE)

class_counts = [6705, 1113, 1099, 514, 327, 142, 115] 
class_weights = [sum(class_counts)/c for c in class_counts]
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
best_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    # --- Training Phase ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_loss_epoch = running_loss / len(train_loader)
    train_acc_epoch = correct / total
        
    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_loss_epoch = val_loss / len(val_loader)
    val_acc_epoch = val_correct / val_total
    
    # Save for local graphs
    history['train_loss'].append(train_loss_epoch)
    history['val_loss'].append(val_loss_epoch)
    history['train_acc'].append(train_acc_epoch)
    history['val_acc'].append(val_acc_epoch)
    
    # NEW: Send metrics to Weights & Biases dashboard!
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss_epoch,
        "train_acc": train_acc_epoch,
        "val_loss": val_loss_epoch,
        "val_acc": val_acc_epoch
    })
    
    print(f"Epoch {epoch+1} | Train Loss: {train_loss_epoch:.4f} | Val Loss: {val_loss_epoch:.4f} | Val Acc: {val_acc_epoch:.4f}")
    
    if val_acc_epoch > best_acc:
        best_acc = val_acc_epoch
        torch.save(model.state_dict(), 'best_model.pth')
        print(" -> Saved new best model")

# NEW: Tell WandB that training is officially complete
wandb.finish()

Epoch 1/15 [Train]:   0%|          | 0/251 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load('best_model.pth', weights_only=True))
model.eval()

# ---------------------------------------------------------
# GRAPH 1: Training & Validation Curves
# ---------------------------------------------------------
epochs_range = range(1, NUM_EPOCHS + 1)
plt.figure(figsize=(12, 5))

# Loss Subplot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss', color='blue')
plt.plot(epochs_range, history['val_loss'], label='Val Loss', color='red')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy Subplot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], label='Train Accuracy', color='blue')
plt.plot(epochs_range, history['val_acc'], label='Val Accuracy', color='red')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# MODEL EVALUATION & PREDICTIONS
# ---------------------------------------------------------
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

target_names = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
print("\nClassification Report:\n", classification_report(all_labels, all_preds, target_names=target_names))

# ---------------------------------------------------------
# GRAPH 2: Confusion Matrix
# ---------------------------------------------------------
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - HAM10000')
plt.ylabel('True Diagnosis')
plt.xlabel('Predicted Diagnosis')
plt.savefig('confusion_matrix_final.png', bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# GRAPH 3: Class Distribution
# ---------------------------------------------------------
df = pd.read_csv(CSV_PATH)
plt.figure(figsize=(10, 6))
sns.countplot(y='dx', data=df, order=df['dx'].value_counts().index, palette='viridis')
plt.title('Class Distribution in HAM10000')
plt.xlabel('Number of Images')
plt.ylabel('Diagnosis Class')
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- FUNCTION TO UN-NORMALIZE IMAGES FOR DISPLAY ---
def imshow(inp, title=None):
    """Convert tensor back to an image so matplotlib can show it"""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title is not None:
        plt.title(title, fontsize=12)

# --- GENERATE PREDICTIONS FOR ONE BATCH ---
# Grab one batch of validation images
inputs, classes = next(iter(val_loader))
inputs = inputs.to(DEVICE)
classes = classes.to(DEVICE)

# Get predictions
outputs = model(inputs)
_, preds = torch.max(outputs, 1)

# --- PLOT THE IMAGES ---
fig = plt.figure(figsize=(15, 10))

# We will display 6 images from the batch
for i in range(6): 
    ax = fig.add_subplot(2, 3, i+1, xticks=[], yticks=[])
    
    # Show the image
    imshow(inputs.cpu().data[i])
    
    # Get the text labels
    true_label = target_names[classes[i].item()]
    pred_label = target_names[preds[i].item()]
    
    # Color the title green if correct, red if incorrect
    color = 'green' if true_label == pred_label else 'red'
    ax.set_title(f'True: {true_label} | Pred: {pred_label}', color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('sample_predictions.png', bbox_inches='tight')
plt.show()

## 5. Exploratory Data Analysis (EDA)
Visualizing the class distribution.

## 6. Model Architecture (Fine-Tuning)
Using **ResNet50** with **Layer 4 Unfrozen**. This allows the model to learn specific skin textures (Fine-Tuning). We also apply **Balanced Class Weights** to handle the imbalance.

## 7. Training Loop
Training for **15 Epochs**.

## 8. Training Visualization (PLT)
Plotting the Learning Curves (Loss and Accuracy over epochs) to check for convergence.

## 9. Validation Metrics
Generating the **Classification Report** and **Confusion Matrix** to evaluate Precision, Recall, and per-class performance.

## 10. Sample Predictions
Visualizing actual model predictions on validation images.